# 🎬 KHEDIM AI — Generate Video · TPU v5 Edition
**Fondé par KHEDIM BENYAKHLEF**

> **Instructions Colab** :
> 1. Menu `Exécution` → `Modifier le type d'exécution` → choisir **TPU v2-8** (ou TPU v5 si disponible)
> 2. Exécuter **Cellule 1** (installation) — attendre ✅
> 3. Exécuter **Cellule 2** (initialisation TPU + modèles) — attendre ✅
> 4. Exécuter **Cellule 3** (serveur) — copier l'URL ngrok dans l'interface

---
**Aucune clé API Anthropic requise** — détection de style par analyse locale.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 1 / 3 — Installation  (exécuter une seule fois)           ║
# ╚══════════════════════════════════════════════════════════════════════╝
import subprocess, sys

def pip(pkg, extra=''):
    cmd = f'pip install -q {extra} {pkg}'
    r   = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=900)
    ok  = '✅' if r.returncode == 0 else '⚠️'
    print(f'  {ok} {pkg.split()[0][:40]}')
    return r.returncode == 0

print('╔══════════════════════════════════════════╗')
print('║  KHEDIM AI — Installation TPU v5/v6e    ║')
print('╚══════════════════════════════════════════╝')
print()

# ── PyTorch CPU + PyTorch-XLA (TPU v5 / v6e Trillium) ────────────────
pip('torch==2.3.1 torchvision==0.18.1',
    '--index-url https://download.pytorch.org/whl/cpu')

pip('torch_xla[tpu]',
    '-f https://storage.googleapis.com/libtpu-releases/index.html')

# ── JAX (utilitaires TPU) ─────────────────────────────────────────────
pip('jax[tpu]',
    '-f https://storage.googleapis.com/jax-releases/libtpu_releases.html')

# ── Diffusion (génération vidéo / image) ──────────────────────────────
for p in [
    'diffusers==0.28.2',
    'transformers==4.41.2',
    'accelerate==0.30.1',
    'peft==0.11.1',
    'safetensors==0.4.3',
    'omegaconf einops',
    'huggingface_hub==0.23.2',
    'compel',
]:
    pip(p)

# ── Vidéo & Image ─────────────────────────────────────────────────────
for p in [
    'imageio==2.34.1 imageio-ffmpeg',
    'opencv-python-headless',
    'Pillow>=10.3.0',
    'moviepy==1.0.3 ffmpeg-python',
    'av',
    'decord',
]:
    pip(p)

# ── Audio IA : Voix (Coqui XTTS) + Musique (MusicGen Meta) ───────────
for p in [
    'TTS==0.22.0',
    'audiocraft',
    'pydub scipy',
    'soundfile',
    'gtts',       # Fallback voix légère
]:
    pip(p)

# ── Backend & utilitaires ─────────────────────────────────────────────
for p in [
    'flask==3.0.3 flask-cors==4.0.1',
    'pyngrok',
    'python-dotenv',
    'numpy==1.26.4',
    'deep-translator',
    'tqdm rich',
]:
    pip(p)

print()
print('✅ Installation terminée — passez à la Cellule 2')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 2 / 3 — Init TPU v5 + Chargement modèles                  ║
# ╚══════════════════════════════════════════════════════════════════════╝
import os, sys, uuid, random, threading, time, gc, json, io
import numpy as np
import torch
import logging

logging.basicConfig(level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s')
log = logging.getLogger('KhedimAI')

print('╔══════════════════════════════════════════╗')
print('║  KHEDIM AI — Init TPU v5                ║')
print('╚══════════════════════════════════════════╝')

# ─── 1. Détection TPU (v5 / v6e / v2-8 / CPU fallback) ──────────────
try:
    import torch_xla.core.xla_model as xm
    device      = xm.xla_device()
    use_tpu     = True
    tpu_cores   = xm.xrt_world_size()
    DTYPE       = torch.bfloat16   # Natif TPU v5 & v6e
    DEVICE_NAME = f'TPU ({tpu_cores} cores)'
    # Test rapide
    _t = torch.ones(4, 4, dtype=DTYPE).to(device)
    xm.mark_step(); del _t
    print(f'  ✅ TPU détecté — {tpu_cores} cœurs — bfloat16')
except ImportError:
    device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    use_tpu     = False
    tpu_cores   = 0
    xm          = None
    DTYPE       = torch.float16 if device.type == 'cuda' else torch.float32
    DEVICE_NAME = str(device).upper()
    print(f'  ⚠️  Fallback: {DEVICE_NAME}')

def mark_step():
    if use_tpu and xm: xm.mark_step()

def free_mem():
    gc.collect(); mark_step()
    if not use_tpu and torch.cuda.is_available():
        torch.cuda.empty_cache()

# ─── 2. Paramètres résolution selon TPU ──────────────────────────────
if tpu_cores >= 32:
    W, H=3840,2160; MAX_FRAMES=480; STEPS=50; CHUNK=24; QUALITY='4K'
elif tpu_cores >= 8:
    W, H=1920,1080; MAX_FRAMES=240; STEPS=45; CHUNK=16; QUALITY='1080p'
elif tpu_cores >= 4:
    W, H=1280,720;  MAX_FRAMES=160; STEPS=40; CHUNK=12; QUALITY='720p'
elif use_tpu:
    W, H=768,432;   MAX_FRAMES=96;  STEPS=35; CHUNK=8;  QUALITY='SD+'
else:
    W, H=512,288;   MAX_FRAMES=48;  STEPS=30; CHUNK=8;  QUALITY='SD'

FPS=8; GUIDANCE=7.5; DEFAULT_FRAMES=min(64, MAX_FRAMES)

print(f'  📐 Résolution : {W}×{H} ({QUALITY}) 16/9')
print(f'  🎞️  Frames max : {MAX_FRAMES}')

# ─── 3. Répertoires ──────────────────────────────────────────────────
BASE = ('/content' if os.path.exists('/content') else
        '/kaggle/working' if os.path.exists('/kaggle/working') else
        os.path.expanduser('~'))
DIRS = {k: os.path.join(BASE, k)
        for k in ['outputs','frames','audio','temp','segments']}
for d in DIRS.values(): os.makedirs(d, exist_ok=True)

# ─── 4. État global ──────────────────────────────────────────────────
import threading as _th
_state = {
    'running':False,'progress':0,'step':'En attente...',
    'current_frame':0,'total_frames':0,
    'video_path':None,'image_path':None,'final_path':None,
    'error':None,'job_id':None,
    'device':DEVICE_NAME,'quality':QUALITY,
    'resolution':f'{W}x{H}','tpu_cores':tpu_cores,
}
_lock = _th.Lock()
def S(**kw):
    with _lock: _state.update(kw)

# ─── 5. Styles visuels (sans API Anthropic) ───────────────────────────
STYLE_KW = {
    'cyberpunk':  ['cyber','neon','dystopi','robot','futur','hack','matrix'],
    'nature':     ['forêt','montagne','océan','nature','jungle','desert','flower','tree'],
    'scifi':      ['espace','vaisseau','alien','galaxie','planète','space','cosmos'],
    'noir':       ['détective','sombre','pluie','noir','mystère','crime','shadow'],
    'fantasy':    ['magie','dragon','elfe','château','enchanté','fantasy','wizard'],
    'horror':     ['horreur','monstre','ombre','terreur','peur','dark','blood'],
    'romantique': ['amour','romance','coucher','lumière dorée','doux','tendresse'],
    'epic_battle':['bataille','guerre','armée','explosion','combat','warrior','fight'],
    'cinematic':  [],
}
STYLE_PROMPTS = {
    'cinematic':   'cinematic lighting, dramatic shadows, anamorphic lens, film grain, ultra realistic, 4K HDR',
    'cyberpunk':   'neon lights, rain-soaked streets, volumetric fog, holographic UI, ultra realistic, 8K, blade runner',
    'nature':      'golden hour, ray of light, ultra wide angle, cinematic grade, 4K HDR, documentary, national geographic',
    'scifi':       'sci-fi technology, galactic nebula, volumetric lighting, ultra detailed, cinematic, IMAX',
    'noir':        'film noir, black and white with accent colors, rain, smoky atmosphere, dramatic shadows, chiaroscuro',
    'fantasy':     'enchanted atmosphere, magical particles, bioluminescent, epic fantasy, ultra detailed, 8K, Tolkien',
    'horror':      'atmospheric fog, moonlight, horror atmosphere, dark shadows, cinematic tension, ultra realistic',
    'romantique':  'golden hour, warm bokeh, emotional depth, dreamy atmosphere, soft cinematic light, 4K romantic',
    'epic_battle': 'epic scale, massive army, cinematic explosion, HDR lighting, ultra realistic, IMAX, 8K',
}
NARRATIONS = {
    'cinematic':   'Dans cette scène épique, chaque instant se grave dans la mémoire collective.',
    'cyberpunk':   'Dans ce futur dystopique, la frontière entre humain et machine s'efface.',
    'nature':      'La nature révèle toute sa splendeur dans un spectacle intemporel.',
    'scifi':       'Aux confins de l'univers, une nouvelle ère de découvertes commence.',
    'noir':        'Dans les ruelles sombres de la ville, les vérités restent enterrées.',
    'fantasy':     'Dans ce royaume enchanté, la magie et le destin s'entrelacent.',
    'horror':      'L'obscurité cache des secrets que l'âme humaine n'est pas prête à affronter.',
    'romantique':  'Dans la lumière dorée de cet instant, l'amour transcende le temps.',
    'epic_battle': 'Le destin du monde se joue dans cette bataille légendaire.',
}
MUSIC_DESCS = {
    'cinematic':   'epic cinematic orchestral music, dramatic, emotional, film score',
    'cyberpunk':   'dark electronic synthwave, industrial beats, cyberpunk atmosphere',
    'nature':      'peaceful ambient nature, birds, soft piano, documentary',
    'scifi':       'space ambient electronic, mysterious, sci-fi atmosphere',
    'noir':        'jazz noir, saxophone, rainy night atmosphere, melancholic',
    'fantasy':     'epic fantasy orchestral, magical, heroic adventure, strings',
    'horror':      'dark ambient horror, tension, dissonant strings, fear',
    'romantique':  'romantic piano, soft strings, intimate, emotional, love theme',
    'epic_battle': 'massive orchestral battle music, drums, brass, epic warfare',
}
MUSIC_TONES = {
    'cinematic':[130,165,196,220,261,329,392],
    'cyberpunk':[110,138,165,220,277,370,440],
    'nature':[261,330,392,494,523,659,784],
    'scifi':[174,220,261,348,440,523,698],
    'noir':[116,146,174,196,233,293,349],
    'fantasy':[196,247,294,370,440,554,659],
    'horror':[87,110,130,164,196,220,246],
    'romantique':[261,330,392,494,587,659,784],
    'epic_battle':[98,130,164,196,261,330,392],
}
NEG = ('blurry, low resolution, bad anatomy, deformed face, ugly, '
       'watermark, text overlay, cartoon, anime, flat colors, '
       'jpeg artifacts, grain noise, distorted, bad proportions')

def detect_style(prompt):
    pl = prompt.lower()
    scores = {s:sum(1 for k in kws if k in pl) for s,kws in STYLE_KW.items()}
    best   = max(scores, key=lambda s: scores[s])
    return best if scores[best] > 0 else 'cinematic'

# ─── 6. AnimateDiff ──────────────────────────────────────────────────
print('\n🎬 Chargement AnimateDiff...')
from diffusers import AnimateDiffPipeline, DDIMScheduler, MotionAdapter

adapter = MotionAdapter.from_pretrained(
    'guoyww/animatediff-motion-adapter-v1-5-2',
    torch_dtype=DTYPE, low_cpu_mem_usage=True
)
pipe_video = AnimateDiffPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    motion_adapter=adapter,
    torch_dtype=DTYPE, low_cpu_mem_usage=True, use_safetensors=True,
)
pipe_video.scheduler = DDIMScheduler.from_pretrained(
    'runwayml/stable-diffusion-v1-5', subfolder='scheduler',
    clip_sample=False, timestep_spacing='linspace',
    beta_schedule='linear', steps_offset=1,
)
pipe_video = pipe_video.to(device)
pipe_video.vae.enable_slicing()
pipe_video.vae.enable_tiling()
pipe_video.enable_attention_slicing()
mark_step()
print(f'  ✅ AnimateDiff → {DEVICE_NAME}')

# ─── 7. SDXL image ───────────────────────────────────────────────────
print('🖼️  Chargement SDXL...')
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

pipe_image = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=DTYPE, use_safetensors=True,
    low_cpu_mem_usage=True, add_watermarker=False,
)
pipe_image.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe_image.scheduler.config,
    use_karras_sigmas=True, algorithm_type='dpmsolver++'
)
pipe_image = pipe_image.to(device)
pipe_image.enable_attention_slicing()
mark_step()
print(f'  ✅ SDXL → {DEVICE_NAME}')

# ─── 8. Voix IA ──────────────────────────────────────────────────────
print('🎙️  Chargement TTS...')
tts_model=None; tts_available=False
try:
    from TTS.api import TTS as CoquiTTS
    os.environ['COQUI_TOS_AGREED']='1'
    tts_model     = CoquiTTS('tts_models/multilingual/multi-dataset/xtts_v2')
    tts_available = True
    print('  ✅ Coqui XTTS v2 prêt')
except Exception as e:
    print(f'  ⚠️  XTTS: {str(e)[:50]}')
    try:
        from gtts import gTTS; tts_available='gtts'
        print('  ✅ gTTS fallback actif')
    except: print('  ❌ Aucun TTS')

VOICE_PROFILES={
    'masculin':  {'speaker':'Claribel Dervla','language':'fr'},
    'feminin':   {'speaker':'Daisy Studious','language':'fr'},
    'dramatique':{'speaker':'Viktor Eka','language':'fr'},
    'epique':    {'speaker':'Andrew Chipper','language':'fr'},
}

# ─── 9. MusicGen ─────────────────────────────────────────────────────
print('🎵 Chargement MusicGen...')
music_model=None; music_available=False
try:
    from audiocraft.models import MusicGen
    music_model     = MusicGen.get_pretrained('facebook/musicgen-medium')
    music_available = True
    print('  ✅ MusicGen Medium prêt')
except Exception as e:
    print(f'  ⚠️  MusicGen: {str(e)[:50]}')

print(f'\n{"="*50}')
print(f'  ✅ KHEDIM AI — TPU PRÊT!')
print(f'  Device  : {DEVICE_NAME}')
print(f'  Qualité : {QUALITY} ({W}×{H})')
print(f'  TTS     : {"XTTS v2" if tts_available is True else "gTTS" if tts_available else "❌"}')
print(f'  Musique : {"MusicGen" if music_available else "synthèse"}')
print(f'{"="*50}')
print('  → Passez à la Cellule 3')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 3 / 3 — Pipeline génération + Serveur Flask + Tunnel       ║
# ╚══════════════════════════════════════════════════════════════════════╝
import subprocess, imageio, scipy.io.wavfile as wavfile
from moviepy.editor import (VideoFileClip, AudioFileClip,
                             CompositeAudioClip, concatenate_audioclips)
from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS
from queue import Queue, Empty

# ── Config ────────────────────────────────────────────────────────────
# Renseignez votre token ngrok : https://dashboard.ngrok.com/auth
NGROK_TOKEN = os.getenv('NGROK_TOKEN', '')  # ou collez-le ici entre ''

# ── Génération voix ───────────────────────────────────────────────────
def gen_voix(text, style='masculin'):
    if not tts_available or not text: return None
    out = os.path.join(DIRS['audio'], f'voix_{uuid.uuid4().hex[:6]}.wav')
    try:
        if tts_available is True:
            p = VOICE_PROFILES.get(style, VOICE_PROFILES['masculin'])
            tts_model.tts_to_file(text=text, speaker=p['speaker'],
                                   language=p['language'], file_path=out)
        else:
            from gtts import gTTS
            mp3 = out.replace('.wav','.mp3')
            gTTS(text=text, lang='fr').save(mp3)
            subprocess.run(['ffmpeg','-y','-i',mp3,'-ar','22050',out],
                           capture_output=True, timeout=30)
        return out if os.path.exists(out) else None
    except Exception as e:
        log.error(f'Voix: {e}'); return None

# ── Génération musique ────────────────────────────────────────────────
def gen_musique(style, duree):
    out = os.path.join(DIRS['audio'], f'music_{uuid.uuid4().hex[:6]}.wav')
    try:
        if music_available:
            desc = MUSIC_DESCS.get(style, MUSIC_DESCS['cinematic'])
            music_model.set_generation_params(
                duration=min(int(duree)+2, 30), temperature=1.0, top_k=250)
            wav = music_model.generate([desc])
            audio_np = wav[0,0].cpu().numpy()
            sr = music_model.sample_rate
            wavfile.write(out, sr, audio_np.astype(np.float32))
        else:
            sr=44100; t=np.linspace(0,duree,int(sr*duree))
            freqs=MUSIC_TONES.get(style,MUSIC_TONES['cinematic'])
            sig=sum((0.18/(i+1))*np.sin(2*np.pi*f*t) for i,f in enumerate(freqs))
            fade=int(sr*2)
            sig[:fade]*=np.linspace(0,1,fade)
            sig[-fade:]*=np.linspace(1,0,fade)
            sig=sig/(np.abs(sig).max()+1e-6)*0.80
            wavfile.write(out, sr, sig.astype(np.float32))
        return out
    except Exception as e:
        log.error(f'Musique: {e}'); return None

# ── Génération image SDXL ─────────────────────────────────────────────
def gen_image(prompt, resolution='1024x1024', steps=35, guidance=7.5,
               seed=-1, style=None):
    if not style: style = detect_style(prompt)
    if seed==-1: seed=random.randint(0,2**31)
    resmap={'4k':(3840,2160),'2k':(2560,1440),'1080p':(1920,1080),
            '720p':(1280,720),'1024x1024':(1024,1024),'512x512':(512,512)}
    iw,ih=resmap.get(resolution.lower(),(1024,1024))
    full=f'{prompt}, {STYLE_PROMPTS[style]}'
    S(step=f'🖼️ Image SDXL {iw}×{ih}...',progress=10)
    gen=torch.Generator('cpu').manual_seed(seed)
    out=pipe_image(prompt=full,negative_prompt=NEG,width=iw,height=ih,
                   num_inference_steps=steps,guidance_scale=guidance,generator=gen)
    mark_step()
    path=os.path.join(DIRS['outputs'],f'image_{uuid.uuid4().hex[:8]}.png')
    out.images[0].save(path, quality=98)
    free_mem()
    S(image_path=path,progress=100,step='✅ Image générée!')
    return path

# ── Génération vidéo longue durée (segments) ──────────────────────────
def gen_video(prompt, style=None, duration_sec=5, steps=None,
              guidance=7.5, fps=FPS, seed=-1,
              voix_active=True, style_voix='masculin',
              texte_voix=None, musique_active=True):
    if not style: style=detect_style(prompt)
    if seed==-1: seed=random.randint(0,2**31)
    steps=steps or STEPS
    num_frames=min(int(duration_sec*fps),MAX_FRAMES)
    n_chunks=max(1,(num_frames+CHUNK-1)//CHUNK)
    full_p=f'{prompt}, {STYLE_PROMPTS[style]}'
    job_id=_state.get('job_id',uuid.uuid4().hex[:8])

    S(total_frames=num_frames,
      step=f'🔥 {QUALITY} — {n_chunks} segments...',progress=2)
    log.info(f'[GEN] {num_frames}f | {W}×{H} | {style} | {DEVICE_NAME}')

    gen=torch.Generator('cpu').manual_seed(seed)
    seg_paths=[]

    for ci in range(n_chunks):
        n_f=min(CHUNK,num_frames-ci*CHUNK)
        pct=int(2+(ci/n_chunks)*52)
        S(step=f'🔥 Segment {ci+1}/{n_chunks} ({n_f} frames)',
          progress=pct, current_frame=ci*CHUNK)
        out=pipe_video(
            prompt=full_p,negative_prompt=NEG,
            num_frames=n_f,width=W,height=H,
            num_inference_steps=steps,guidance_scale=guidance,generator=gen)
        mark_step()  # ← OBLIGATOIRE sur TPU
        seg=os.path.join(DIRS['segments'],f'seg_{job_id}_{ci:04d}.mp4')
        writer=imageio.get_writer(seg,fps=fps,codec='libx264',
                                   quality=9,macro_block_size=1)
        for frame in out.frames[0]:
            writer.append_data(np.array(frame))
        writer.close()
        seg_paths.append(seg)
        free_mem()

    S(step='🎬 Assemblage...',progress=55)
    list_file=os.path.join(DIRS['temp'],f'list_{job_id}.txt')
    with open(list_file,'w') as f:
        for sp in seg_paths: f.write(f"file '{sp}'\n")
    vid_raw=os.path.join(DIRS['outputs'],f'video_{job_id}.mp4')
    subprocess.run(['ffmpeg','-y','-f','concat','-safe','0',
                    '-i',list_file,'-c','copy',vid_raw],
                   capture_output=True,check=True)
    for sp in seg_paths:
        try: os.remove(sp)
        except: pass

    S(video_path=vid_raw,progress=57,step='🎙️ Audio IA...')
    video_clip=VideoFileClip(vid_raw)
    duree=video_clip.duration
    tracks=[]

    if musique_active:
        S(step='🎵 MusicGen...',progress=65)
        mp=gen_musique(style,duree)
        if mp:
            raw=AudioFileClip(mp).volumex(0.30)
            if raw.duration<duree:
                n=int(duree/raw.duration)+1
                raw=concatenate_audioclips([raw]*n)
            tracks.append(raw.subclip(0,duree))

    if voix_active:
        S(step='🎙️ Voix off XTTS...',progress=78)
        txt=texte_voix or NARRATIONS.get(style,NARRATIONS['cinematic'])
        vp=gen_voix(txt,style_voix)
        if vp:
            vc=AudioFileClip(vp).volumex(0.88)
            if vc.duration>duree: vc=vc.subclip(0,duree)
            tracks.append(vc)

    S(step='📽️ Encodage final HQ...',progress=88)
    final_path=os.path.join(DIRS['outputs'],f'KHEDIM_FINAL_{job_id}.mp4')
    tmp_audio=os.path.join(DIRS['temp'],f'audio_{job_id}.m4a')

    if tracks:
        audio_final=(CompositeAudioClip(tracks) if len(tracks)>1 else tracks[0])
        video_clip.set_audio(audio_final).write_videofile(
            final_path,temp_audiofile=tmp_audio,remove_temp=True,
            codec='libx264',audio_codec='aac',fps=fps,
            bitrate='8000k',audio_bitrate='320k',logger=None)
    else:
        video_clip.write_videofile(final_path,codec='libx264',
                                    fps=fps,bitrate='8000k',logger=None)
    video_clip.close()
    S(final_path=final_path,progress=100,step='✅ Vidéo terminée!')
    return final_path

# ── Flask app ─────────────────────────────────────────────────────────
kapp=Flask('KhedimAI')
CORS(kapp,resources={r'/*':{'origins':'*'}})
jq=Queue(maxsize=5); js={}; jl=threading.Lock()

def enqueue(job):
    if jq.full(): return None,'File pleine'
    jid=uuid.uuid4().hex[:8]
    job['job_id']=jid
    S(job_id=jid,running=False,progress=0,step='En file...')
    with jl: js[jid]={'status':'queued','progress':0,'step':'En attente...',
                       'result':None,'error':None,'type':job.get('type'),
                       'device':DEVICE_NAME,'quality':QUALITY}
    jq.put(job); return jid,None

@kapp.route('/health')
def health():
    return jsonify({'ok':True,'version':'13.0-tpu5',
                    'device':DEVICE_NAME,'quality':QUALITY,
                    'resolution':f'{W}x{H}','tpu_cores':tpu_cores,
                    'use_tpu':use_tpu,'dtype':str(DTYPE),
                    'tts':'xtts_v2' if tts_available is True else ('gtts' if tts_available else 'off'),
                    'music':'musicgen' if music_available else 'synth',
                    'running':_state['running'],'progress':_state['progress'],
                    'step':_state['step']})

@kapp.route('/progress')
def progress_route():
    with _lock: s=dict(_state)
    return jsonify(s)

@kapp.route('/generate',methods=['POST'])
def route_gen():
    d=request.get_json(force=True)
    prompt=d.get('prompt','').strip()
    if not prompt: return jsonify({'error':'Prompt vide'}),400
    job={'type':'video','prompt':prompt,
         'style':d.get('style'),
         'duration_sec':min(int(d.get('duration_sec',5)),600),
         'steps':min(int(d.get('steps',STEPS)),60),
         'guidance':float(d.get('guidance',GUIDANCE)),
         'fps':int(d.get('fps',FPS)),
         'seed':int(d.get('seed',-1)),
         'voix_active':bool(d.get('voix_active',True)),
         'style_voix':d.get('style_voix','masculin'),
         'texte_voix':d.get('texte_voix'),
         'musique_active':bool(d.get('musique_active',True))}
    jid,err=enqueue(job)
    if err: return jsonify({'error':err}),503
    return jsonify({'job_id':jid,'status':'queued','device':DEVICE_NAME,'quality':QUALITY})

@kapp.route('/generate_image',methods=['POST'])
def route_img():
    d=request.get_json(force=True)
    prompt=d.get('prompt','').strip()
    if not prompt: return jsonify({'error':'Prompt vide'}),400
    job={'type':'image','prompt':prompt,'style':d.get('style'),
         'resolution':d.get('resolution','1024x1024'),
         'steps':min(int(d.get('steps',35)),60),
         'guidance':float(d.get('guidance',7.5)),
         'seed':int(d.get('seed',-1))}
    jid,err=enqueue(job)
    if err: return jsonify({'error':err}),503
    return jsonify({'job_id':jid,'status':'queued'})

@kapp.route('/status/<jid>')
def route_status(jid):
    with jl: j=js.get(jid)
    if not j: return jsonify({'error':'Job inconnu'}),404
    return jsonify(j)

@kapp.route('/final/<fname>')
def serve_final(fname): return send_from_directory(DIRS['outputs'],fname)

@kapp.route('/image/<fname>')
def serve_image(fname): return send_from_directory(DIRS['outputs'],fname)

# ── Worker thread ─────────────────────────────────────────────────────
def worker():
    while True:
        try: job=jq.get(timeout=5)
        except Empty: continue
        jid=job['job_id']
        with jl: js[jid]['status']='processing'
        S(running=True,job_id=jid,progress=0)
        try:
            if job['type']=='video':
                final=gen_video(**{k:v for k,v in job.items()
                                   if k not in ('type','job_id')})
                url=f'/final/{os.path.basename(final)}'
                with jl: js[jid].update({'status':'done','result':url,
                                          'progress':100,'step':'✅ Terminé!'})
            elif job['type']=='image':
                img=gen_image(**{k:v for k,v in job.items()
                                 if k not in ('type','job_id')})
                url=f'/image/{os.path.basename(img)}'
                with jl: js[jid].update({'status':'done','result':url,
                                          'progress':100,'step':'✅ Image prête!'})
        except Exception as e:
            import traceback; traceback.print_exc()
            S(running=False,error=str(e))
            with jl: js[jid].update({'status':'error','error':str(e)})
        finally:
            S(running=False); jq.task_done()

threading.Thread(target=worker,daemon=True,name='KhedimAI-TPU').start()

# ── Tunnel ngrok ──────────────────────────────────────────────────────
PUBLIC_URL=None
if NGROK_TOKEN:
    try:
        from pyngrok import ngrok
        ngrok.set_auth_token(NGROK_TOKEN)
        tunnel=ngrok.connect(8765,bind_tls=True)
        PUBLIC_URL=tunnel.public_url
        print(f'\n{"="*60}')
        print(f'  🌐 URL publique KHEDIM AI : {PUBLIC_URL}')
        print(f'  📋 Collez cette URL dans l\'interface KHEDIM AI')
        print(f'  🔥 Device  : {DEVICE_NAME}')
        print(f'  📐 Qualité : {QUALITY} ({W}×{H})')
        print(f'{"="*60}\n')
    except Exception as e:
        print(f'  ⚠️  Ngrok: {e}')
else:
    print('\n  ⚠️  NGROK_TOKEN non défini — ajoutez-le dans la variable NGROK_TOKEN')
    print('  ℹ️  Obtenez-le sur https://dashboard.ngrok.com/auth')

print('🚀 Serveur KHEDIM AI sur port 8765...')
kapp.run(host='0.0.0.0', port=8765, debug=False, threaded=True)